# SMS Shield — Text Classification Case Study

**Domain:** Text Classification  
**Algorithm:** Multinomial Naive Bayes  
**Feature extraction:** TF-IDF  

Run cells from top to bottom. The first data-loading cell needs internet access to download the UCI dataset.

## 1. Import libraries
We import pandas for tables, matplotlib/seaborn for visualizations and scikit-learn for splitting, text features, training and evaluation.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

## 2. Download and load the UCI dataset
This cell calls the project's downloader, which uses the official UCI source and a mirror fallback.

In [ ]:
from src_data_loader import load_and_clean_dataset

df = load_and_clean_dataset()
df.head()

## 3. Understand and validate the dataset

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate messages:', df['message'].duplicated().sum())
print('\nClass distribution:\n', df['label'].value_counts())

## 4. Visualize the class distribution
The classes are not equally represented, so we use a stratified split and report precision/recall/F1 in addition to accuracy.

In [ ]:
sns.set_theme(style='whitegrid')
ax = df['label'].value_counts().reindex(['ham', 'spam']).plot(kind='bar', color=['#6B8FD6', '#E58B82'])
ax.set_title('SMS Class Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Number of messages')
plt.tight_layout()
plt.show()

## 5. Separate input text and target labels

In [ ]:
X = df['message']
y = df['label']
print('Input samples:', len(X))
print('Target classes:', sorted(y.unique()))

## 6. Train-test split (80:20)
`stratify=y` preserves class proportions. `random_state=42` makes the split reproducible. TF-IDF will be fitted only after this split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Training records:', len(X_train))
print('Testing records:', len(X_test))

## 7. Build TF-IDF + Multinomial Naive Bayes pipeline
TF-IDF extracts unigrams and bigrams. Stop words are retained to preserve negations. Pipeline prevents preprocessing leakage.

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True, strip_accents='unicode', ngram_range=(1, 2),
        min_df=2, max_df=0.98, sublinear_tf=True
    )),
    ('classifier', MultinomialNB(alpha=0.5))
])

## 8. Train the model

In [ ]:
model.fit(X_train, y_train)
print('Training completed.')

## 9. Predict on unseen test messages

In [ ]:
y_pred = model.predict(X_test)
print('First 10 predictions:', y_pred[:10])

## 10. Evaluate the classifier
The lab requires confusion matrix, accuracy, precision, recall and F1-score. We report spam-class and weighted metrics.

In [ ]:
labels = ['ham', 'spam']
cm = confusion_matrix(y_test, y_pred, labels=labels)
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision_spam': precision_score(y_test, y_pred, pos_label='spam', zero_division=0),
    'recall_spam': recall_score(y_test, y_pred, pos_label='spam', zero_division=0),
    'f1_spam': f1_score(y_test, y_pred, pos_label='spam', zero_division=0),
    'precision_weighted': precision_score(y_test, y_pred, average='weighted', zero_division=0),
    'recall_weighted': recall_score(y_test, y_pred, average='weighted', zero_division=0),
    'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
}
print(json.dumps({k: round(float(v), 4) for k, v in metrics.items()}, indent=2))
print('\nClassification report:\n', classification_report(y_test, y_pred, labels=labels, zero_division=0))

## 11. Confusion matrix (light theme)

In [ ]:
plt.figure(figsize=(6, 5), facecolor='white')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title('SMS Spam Detection - Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('Actual label')
plt.tight_layout()
plt.show()

## 12. Test a custom message
Try a few messages and compare the output. These examples are live demonstrations, not part of the held-out test metrics.

In [ ]:
examples = [
    'Congratulations! You have won a free prize. Call now to claim.',
    'Hey, are we still meeting for lunch today?',
]
for text in examples:
    print(text, '=>', model.predict([text])[0])

## 13. Save the trained model and metrics
The saved pipeline contains both TF-IDF and Naive Bayes, so the application uses exactly the same preprocessing as training.

In [ ]:
from pathlib import Path
import joblib
ROOT = Path.cwd()
(ROOT / 'models').mkdir(exist_ok=True)
(ROOT / 'outputs').mkdir(exist_ok=True)
joblib.dump(model, ROOT / 'models' / 'sms_spam_pipeline.joblib')
final_metrics = {k: float(v) for k, v in metrics.items()}
final_metrics.update({'dataset_records_after_cleaning': int(len(df)), 'train_records': int(len(X_train)), 'test_records': int(len(X_test)), 'random_state': 42, 'test_size': 0.20})
(ROOT / 'outputs' / 'metrics.json').write_text(json.dumps(final_metrics, indent=4), encoding='utf-8')
print('Saved model and metrics.')

## 14. Conclusion and limitations
The model learns statistical associations between message terms and spam/ham labels. It is an educational baseline, not a guaranteed security filter. The historical English-language dataset may not represent modern, multilingual or regional spam. Future work can compare SVM/Logistic Regression and evaluate newer datasets.